# 1. Preparação do Ambiente e Verificação de GPU

In [ ]:
# CÉLULA 1: Estrutura de Pastas e GPU
!rm -rf /tmp/bonsai-demo /content/bonsai/logs
!mkdir -p /content/bonsai/bin /content/bonsai/models /content/bonsai/logs
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

# 2. Motor PrismML, Cloudflared e Download do Modelo (5.95 GB)

In [ ]:
%%bash
# CÉLULA 2: Motores de Inferência e Modelo PTQ1_0

# 1. Garante que os diretórios existam
mkdir -p /content/bonsai/bin /content/bonsai/models /content/bonsai/logs

# 2. Binários Oficiais da PrismML (Fork CUDA)
if [ ! -f "/content/bonsai/bin/cuda/llama-server" ]; then
    echo ">>> [1/3] Baixando motor oficial PrismML..."
    git clone --depth 1 https://github.com/PrismML-Eng/Bonsai-demo.git /tmp/bonsai-demo
    cd /tmp/bonsai-demo
    bash scripts/download_binaries.sh
    cp -r bin/* /content/bonsai/bin/
    rm -rf /tmp/bonsai-demo
    echo "    [OK] Binários instalados com sucesso."
else
    echo ">>> [1/3] Binários já presentes."
fi

# 3. Binário Standalone do Cloudflare Tunnel
if [ ! -f "/content/bonsai/bin/cloudflared" ]; then
    echo ">>> [2/3] Baixando Cloudflared..."
    wget -q -O /content/bonsai/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    chmod +x /content/bonsai/bin/cloudflared
    echo "    [OK] Cloudflared pronto."
else
    echo ">>> [2/3] Cloudflared já presente."
fi

# 4. Download Sequencial dos Pesos PTQ1_0 (~5.95 GB)
MODEL_PATH="/content/bonsai/models/Ternary-Bonsai-2-27B-PTQ1_0.gguf"
if [ ! -f "$MODEL_PATH" ] || [ $(stat -c%s "$MODEL_PATH" 2>/dev/null || echo 0) -lt 5500000000 ]; then
    echo ">>> [3/3] Baixando Ternary Bonsai 2 27B (PTQ1_0)..."
    curl -fL --progress-bar -C - \
      "https://huggingface.co/prism-ml/Ternary-Bonsai-2-27B-gguf/resolve/main/Ternary-Bonsai-2-27B-PTQ1_0.gguf" \
      -o "$MODEL_PATH"
else
    echo ">>> [3/3] Modelo já baixado e íntegro no disco."
fi

echo ">>> Arquivo pronto no disco:"
ls -lh "$MODEL_PATH"

# 3. Inicialização do Servidor e URL Pública Cloudflare

In [ ]:
# CÉLULA 3: Inicialização Definitiva na GPU + Túnel Cloudflare
import os
import re
import sys
import glob
import time
import subprocess
import requests

BASE_DIR = "/content/bonsai"
MODEL_PATH = f"{BASE_DIR}/models/Ternary-Bonsai-2-27B-PTQ1_0.gguf"
BIN_PATH = f"{BASE_DIR}/bin/cuda/llama-server"
CLOUDFLARED_BIN = f"{BASE_DIR}/bin/cloudflared"

# Garante o cuBLAS e Runtime
!pip install -q nvidia-cuda-runtime-cu12 nvidia-cublas-cu12

# Limpa instâncias anteriores
!pkill -f llama-server 2>/dev/null || true
!pkill -f cloudflared 2>/dev/null || true
time.sleep(1)

# Caminhos das bibliotecas CUDA 12
nvidia_pip_libs = glob.glob("/usr/local/lib/python*/dist-packages/nvidia/*/lib")
system_cuda_libs = (
    glob.glob("/usr/local/cuda*/targets/x86_64-linux/lib") +
    glob.glob("/usr/local/cuda*/lib64") +
    glob.glob("/usr/lib64-nvidia") +
    [f"{BASE_DIR}/bin/cuda", f"{BASE_DIR}/bin"]
)

env = os.environ.copy()
env["LD_LIBRARY_PATH"] = ":".join(list(dict.fromkeys(nvidia_pip_libs + system_cuda_libs))) + f":{env.get('LD_LIBRARY_PATH', '')}"

PORT = 8081

server_cmd = [
    BIN_PATH,
    "-m", MODEL_PATH,
    "-ngl", "99",               # 100% offload na GPU T4
    "-fa", "on",                # Flash Attention
    "-c", "8192",               # Contexto seguro contra OOM
    "-n", "16384",              # Teto alto para pensamento
    "-ctk", "q8_0",             # Cache KV quantizado
    "-ctv", "q8_0",
    "--jinja",                  # Template Jinja nativo Qwen3.8
    "--temp", "1.0",
    "--top-p", "0.95",
    "--top-k", "20",
    "--min-p", "0.0",
    "--presence-penalty", "0.0",
    "--repeat-penalty", "1.0",
    "--host", "127.0.0.1",
    "--port", str(PORT)
]

print("Carregando Bonsai 2 27B na VRAM da Tesla T4 (aguarde cerca de 2 min)...")
server_log = open(f"{BASE_DIR}/logs/server.log", "w")
server_proc = subprocess.Popen(server_cmd, env=env, stdout=server_log, stderr=subprocess.STDOUT)

server_ready = False
for _ in range(75):
    if server_proc.poll() is not None:
        print("\n[ERRO] O servidor encerrou inesperadamente! Log:")
        with open(f"{BASE_DIR}/logs/server.log", "r") as f:
            print(f.read())
        sys.exit(1)
    try:
        if requests.get(f"http://127.0.0.1:{PORT}/health", timeout=1).status_code == 200:
            server_ready = True
            break
    except requests.exceptions.RequestException:
        pass
    time.sleep(2)
    print(".", end="", flush=True)

if not server_ready:
    print("\n[TIMEOUT] Tempo limite excedido.")
    sys.exit(1)

print("\n\n" + "="*70)
print("🚀 BONSAI 2 27B CARREGADO NA VRAM COM SUCESSO!")
print("="*70)

print("\n>>> Conectando túnel Cloudflare...")
tunnel_proc = subprocess.Popen([CLOUDFLARED_BIN, "tunnel", "--url", f"http://127.0.0.1:{PORT}"], stderr=subprocess.PIPE, universal_newlines=True)

public_url = None
for line in iter(tunnel_proc.stderr.readline, ''):
    match = re.search(r'https://[-a-zA-Z0-9]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break

print("\n" + "="*70)
print(f"🔗 URL Pública (API OpenAI): {public_url}")
print(f"📡 Endpoint de Chat:       {public_url}/v1/chat/completions")
print("="*70)

# 4. Chat Interativo Local (Com Streaming de Pensamento)

In [ ]:
# CÉLULA 4 (Com Velocímetro de Tokens/s)
import json
import time
import requests

def chat(prompt, system_prompt="Você é um assistente técnico de alta precisão."):
    url = "http://127.0.0.1:8081/v1/chat/completions"
    headers = {"Content-Type": "application/json"}
    payload = {
        "model": "bonsai-2-27b",
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt}
        ],
        "temperature": 1.0,
        "top_p": 0.95,
        "max_tokens": 4096,
        "stream": True,
        "chat_template_kwargs": {"reasoning_effort": "medium"}
    }

    start_time = time.time()
    response = requests.post(url, headers=headers, json=payload, stream=True)

    print("\n" + "="*50)
    in_thinking = False
    token_count = 0
    first_token_time = None

    for line in response.iter_lines():
        if line:
            line_str = line.decode('utf-8')
            if line_str.startswith("data: ") and line_str != "data: [DONE]":
                chunk = json.loads(line_str[6:])
                delta = chunk['choices'][0]['delta']

                # Marca o tempo do primeiro token (fim do prefill)
                if first_token_time is None:
                    first_token_time = time.time()

                # Exibe o raciocínio interno
                if 'reasoning_content' in delta and delta['reasoning_content']:
                    if not in_thinking:
                        print("🤔 PENSAMENTO:\n", end="", flush=True)
                        in_thinking = True
                    print(delta['reasoning_content'], end="", flush=True)
                    token_count += 1

                # Exibe a resposta final
                if 'content' in delta and delta['content']:
                    if in_thinking:
                        print("\n\n💬 RESPOSTA FINAL:\n", end="", flush=True)
                        in_thinking = False
                    print(delta['content'], end="", flush=True)
                    token_count += 1

    total_time = time.time() - start_time
    gen_time = (time.time() - first_token_time) if first_token_time else total_time
    tps = token_count / gen_time if gen_time > 0 else 0

    print("\n" + "="*50)
    print(f"📊 MÉTRICAS: {token_count} tokens gerados em {gen_time:.2f}s | Velocidade: {tps:.2f} tok/s")
    print("="*50 + "\n")

chat("Explique como a computação quântica pode impactar a criptografia de chave pública.")